## Sequence Input Design

Construct 30-day chronological listening sequences for every user.
Preserve separate indicators for log presence and duration validity.

Use transaction features as additional model inputs.
Fit learned preprocessing parameters on the historical training split only.

In [1]:
from pathlib import Path
import platform
import shutil

import numpy as np
import pandas as pd
import duckdb
import torch

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
REPORT_DIR = PROJECT_ROOT / "reports"

print("Python version:", platform.python_version())
print("PyTorch version:", torch.__version__)
print("Machine architecture:", platform.machine())
print("MPS available:", torch.backends.mps.is_available())
print("CUDA available:", torch.cuda.is_available())

print(
    "Available disk space:",
    f"{shutil.disk_usage(PROCESSED_DIR).free / 1024**3:.2f} GiB",
)

Python version: 3.11.15
PyTorch version: 2.14.1
Machine architecture: arm64
MPS available: True
CUDA available: False
Available disk space: 11.81 GiB


In [2]:
sequence_channels = [
    "log1p_num_25",
    "log1p_num_50",
    "log1p_num_75",
    "log1p_num_985",
    "log1p_num_100",
    "log1p_num_unq",
    "log1p_valid_seconds",
    "log_present",
    "duration_valid",
]

SEQUENCE_LENGTH = 30
SEQUENCE_DTYPE = np.float32

cohort_summary = pd.read_csv(
    REPORT_DIR / "combined_feature_summary.csv"
)

sequence_plan = cohort_summary[["cohort", "users"]].copy()

sequence_plan["sequence_days"] = SEQUENCE_LENGTH
sequence_plan["channels"] = len(sequence_channels)

sequence_plan["estimated_array_gib"] = (
    sequence_plan["users"]
    * SEQUENCE_LENGTH
    * len(sequence_channels)
    * np.dtype(SEQUENCE_DTYPE).itemsize
    / 1024**3
)

print(
    "Estimated sequence array size:",
    f"{sequence_plan['estimated_array_gib'].sum():.2f} GiB",
)

sequence_plan

Estimated sequence array size: 1.86 GiB


,cohort,users,sequence_days,channels,estimated_array_gib
0,historical_train,882127,30,9,0.887268
1,later_evaluation,970960,30,9,0.976619


## Encode Daily Records and Verify Alignment

Map records to fixed user and date positions.
Apply log1p to nonnegative numeric inputs.
Preserve separate log-presence and duration-validity indicators.

In [3]:
count_columns = [
    "num_25",
    "num_50",
    "num_75",
    "num_985",
    "num_100",
    "num_unq",
]

def encode_daily_records(frame):
    counts = frame[count_columns].to_numpy(dtype=np.float32)

    assert np.isfinite(counts).all()
    assert (counts >= 0).all()

    duration_valid = frame["total_secs_clean"].notna().to_numpy()

    seconds = (
        frame["total_secs_clean"]
        .fillna(0)
        .to_numpy(dtype=np.float32)
    )

    assert np.isfinite(seconds).all()
    assert (seconds >= 0).all()
    assert (seconds <= 86400).all()

    encoded = np.zeros(
        (len(frame), len(sequence_channels)),
        dtype=np.float32,
    )

    encoded[:, :6] = np.log1p(counts)
    encoded[:, 6] = np.log1p(seconds)
    encoded[:, 7] = 1.0
    encoded[:, 8] = duration_valid.astype(np.float32)

    assert np.isfinite(encoded).all()

    return encoded

def write_daily_records(sequence_array, user_index, window_start, frame):
    assert not frame.duplicated(["msno", "log_day"]).any()

    user_positions = user_index.get_indexer(frame["msno"])

    day_positions = (
        pd.to_datetime(frame["log_day"])
        - pd.Timestamp(window_start)
    ).dt.days.to_numpy(dtype=np.int64)

    assert (user_positions >= 0).all()
    assert (day_positions >= 0).all()
    assert (day_positions < sequence_array.shape[1]).all()

    assert not sequence_array[
        user_positions, day_positions, 7
    ].any(), "A user-day position has already been written."

    sequence_array[
        user_positions, day_positions, :
    ] = encode_daily_records(frame)

In [4]:
toy_user_index = pd.Index(["user_a", "user_b", "user_c"])

toy_sequences = np.zeros(
    (3, SEQUENCE_LENGTH, len(sequence_channels)),
    dtype=np.float32,
)

toy_records = pd.DataFrame({
    "msno": ["user_b", "user_a"],
    "log_day": pd.to_datetime(["2016-11-03", "2016-11-01"]),
    "total_secs_clean": [np.nan, 60.0],
})

for column in count_columns:
    toy_records[column] = 0

toy_records["num_100"] = [5, 3]
toy_records["num_unq"] = [5, 3]

write_daily_records(
    toy_sequences,
    toy_user_index,
    "2016-11-01",
    toy_records,
)

assert toy_sequences[0, 0, 7] == 1
assert toy_sequences[0, 0, 8] == 1
assert np.isclose(toy_sequences[0, 0, 6], np.log1p(60))

assert toy_sequences[1, 2, 7] == 1
assert toy_sequences[1, 2, 8] == 0
assert toy_sequences[1, 2, 6] == 0
assert np.isclose(toy_sequences[1, 2, 4], np.log1p(5))

assert (toy_sequences[0, 1] == 0).all()
assert (toy_sequences[2] == 0).all()

print("Sequence alignment and missingness checks passed.")

toy_summary = pd.DataFrame({
    "user": toy_user_index,
    "recorded_days": toy_sequences[:, :, 7].sum(axis=1),
    "valid_duration_days": toy_sequences[:, :, 8].sum(axis=1),
})

toy_summary

Sequence alignment and missingness checks passed.


,user,recorded_days,valid_duration_days
0,user_a,1.0,1.0
1,user_b,1.0,0.0
2,user_c,0.0,0.0


In [5]:
import json

SEQUENCE_DIR = PROCESSED_DIR / "gru_sequences_v1"
SEQUENCE_DIR.mkdir(parents=True, exist_ok=True)

daily_config = json.loads(
    (REPORT_DIR / "daily_listening_config.json").read_text(
        encoding="utf-8"
    )
)

logs_dir = PROJECT_ROOT / daily_config["output_directory"]
logs_pattern = str(logs_dir / "**" / "*.parquet")

expected_summary = pd.read_csv(
    REPORT_DIR / "listening_window_summary.csv"
).set_index("cohort")

transaction_columns = [
    "transaction_rows_90d",
    "transaction_days_90d",
    "cancel_rows_90d",
    "auto_renew_share_90d",
    "avg_plan_days_90d",
    "days_since_last_transaction_90d",
    "has_transaction_90d",
]

cohort_specs = {
    "historical_train": {
        "user_table": "historical_combined_features_20161130.parquet",
        "window_start": daily_config["training_window"][0],
        "window_end": daily_config["training_window"][1],
    },
    "later_evaluation": {
        "user_table": "later_combined_features_20170228.parquet",
        "window_start": daily_config["evaluation_window"][0],
        "window_end": daily_config["evaluation_window"][1],
    },
}

temp_dir = PROJECT_ROOT / "data" / "duckdb_tmp"
temp_dir.mkdir(parents=True, exist_ok=True)

sequence_con = duckdb.connect(config={
    "threads": 2,
    "memory_limit": "2GB",
    "temp_directory": str(temp_dir),
    "max_temp_directory_size": "2GB",
})

sequence_query = """
SELECT
    msno,
    log_day,
    num_25,
    num_50,
    num_75,
    num_985,
    num_100,
    num_unq,
    total_secs_clean
FROM read_parquet(?, hive_partitioning = true)
WHERE cohort = ?
"""

In [6]:
def build_cohort_sequences(cohort, spec):
    user_path = PROCESSED_DIR / spec["user_table"]

    users = (
        sequence_con.read_parquet(str(user_path))
        .project(", ".join(["msno", "is_churn"] + transaction_columns))
        .df()
        .sort_values("msno")
        .reset_index(drop=True)
    )

    assert users["msno"].is_unique
    assert users["is_churn"].isin([0, 1]).all()

    cohort_dir = SEQUENCE_DIR / cohort
    checkpoint_path = cohort_dir / "_SUCCESS.json"

    source_files = sorted(
        (logs_dir / f"cohort={cohort}").rglob("*.parquet")
    )
    assert source_files, "No listening partitions found."

    configuration = {
        "encoding_version": 1,
        "cohort": cohort,
        "users": len(users),
        "window_start": spec["window_start"],
        "window_end": spec["window_end"],
        "sequence_length": SEQUENCE_LENGTH,
        "sequence_channels": sequence_channels,
        "transaction_columns": transaction_columns,
        "dtype": "float32",
        "user_order": "msno ascending",
        "user_source_size": user_path.stat().st_size,
        "user_source_mtime_ns": user_path.stat().st_mtime_ns,
        "listening_sources": [
            {
                "file": str(path.relative_to(PROJECT_ROOT)),
                "size": path.stat().st_size,
                "mtime_ns": path.stat().st_mtime_ns,
            }
            for path in source_files
        ],
    }

    if checkpoint_path.exists():
        saved_configuration = json.loads(
            checkpoint_path.read_text(encoding="utf-8")
        )
        assert saved_configuration == configuration, (
            "Existing sequence configuration differs from this run."
        )
        print(f"Reusing completed sequences for {cohort}.")
    else:
        assert not cohort_dir.exists(), (
            "An incomplete output directory exists. "
            "Inspect it before rebuilding."
        )

        cohort_dir.mkdir(parents=True)

        sequences = np.lib.format.open_memmap(
            cohort_dir / "sequences.npy",
            mode="w+",
            dtype=np.float32,
            shape=(len(users), SEQUENCE_LENGTH, len(sequence_channels)),
        )

        sequences[:] = 0
        user_index = pd.Index(users["msno"])

        cursor = sequence_con.execute(
            sequence_query,
            [logs_pattern, cohort],
        )

        processed_rows = 0
        next_progress = 1_000_000

        while True:
            batch = cursor.fetch_df_chunk(32)

            if batch.empty:
                break

            write_daily_records(
                sequences,
                user_index,
                spec["window_start"],
                batch,
            )

            processed_rows += len(batch)

            if processed_rows >= next_progress:
                print(
                    f"{cohort}: {processed_rows:,} records written",
                    flush=True,
                )
                next_progress += 1_000_000

        recorded_days = sequences[:, :, 7].sum(axis=1, dtype=np.int32)
        valid_days = sequences[:, :, 8].sum(axis=1, dtype=np.int32)

        expected = expected_summary.loc[cohort]

        assert int(recorded_days.sum()) == processed_rows
        assert processed_rows == int(expected["rows"])
        assert int((recorded_days > 0).sum()) == int(
            expected["users_with_logs"]
        )
        assert int((recorded_days - valid_days).sum()) == int(
            expected["flagged_duration_rows"]
        )
        assert (valid_days <= recorded_days).all()

        transaction_inputs = users[transaction_columns].to_numpy(
            dtype=np.float32,
            na_value=np.nan,
        )

        assert not np.isinf(transaction_inputs).any()

        np.save(
            cohort_dir / "transaction_inputs.npy",
            transaction_inputs,
        )

        np.save(
            cohort_dir / "labels.npy",
            users["is_churn"].to_numpy(dtype=np.float32),
        )

        sequence_con.from_df(
            users[["msno", "is_churn"]]
        ).write_parquet(str(cohort_dir / "users.parquet"))

        sequences.flush()
        del sequences

        checkpoint_path.write_text(
            json.dumps(configuration, indent=2) + "\n",
            encoding="utf-8",
        )

        print(f"Completed {cohort}.")

    sequences = np.load(
        cohort_dir / "sequences.npy",
        mmap_mode="r",
    )

    labels = np.load(
        cohort_dir / "labels.npy",
        mmap_mode="r",
    )

    transactions = np.load(
        cohort_dir / "transaction_inputs.npy",
        mmap_mode="r",
    )

    assert sequences.shape == (
        len(users), SEQUENCE_LENGTH, len(sequence_channels)
    )
    assert sequences.dtype == np.float32
    assert transactions.shape == (len(users), len(transaction_columns))
    assert np.array_equal(
        labels,
        users["is_churn"].to_numpy(dtype=np.float32),
    )

    recorded_days = sequences[:, :, 7].sum(axis=1, dtype=np.int32)
    valid_days = sequences[:, :, 8].sum(axis=1, dtype=np.int32)

    return {
        "cohort": cohort,
        "users": len(users),
        "sequence_days": sequences.shape[1],
        "channels": sequences.shape[2],
        "users_with_logs": int((recorded_days > 0).sum()),
        "users_without_logs": int((recorded_days == 0).sum()),
        "recorded_user_days": int(recorded_days.sum()),
        "invalid_duration_days": int((recorded_days - valid_days).sum()),
    }

In [7]:
required_bytes = int(
    sequence_plan["estimated_array_gib"].sum() * 1024**3
)

assert shutil.disk_usage(PROCESSED_DIR).free > (
    required_bytes + 2 * 1024**3
), "Insufficient working space for sequence construction."

sequence_summaries = []

for cohort, spec in cohort_specs.items():
    print(f"Preparing {cohort}...", flush=True)

    sequence_summaries.append(
        build_cohort_sequences(cohort, spec)
    )

sequence_summary = pd.DataFrame(sequence_summaries)

print(
    "Remaining disk space:",
    f"{shutil.disk_usage(PROCESSED_DIR).free / 1024**3:.2f} GiB",
)

sequence_summary

Preparing historical_train...
historical_train: 1,045,883 records written
historical_train: 2,026,252 records written
historical_train: 3,007,592 records written
historical_train: 4,055,674 records written
historical_train: 5,037,637 records written
historical_train: 6,019,280 records written
historical_train: 7,000,439 records written
historical_train: 8,047,042 records written
historical_train: 9,028,354 records written
historical_train: 10,008,011 records written
historical_train: 11,055,601 records written
Completed historical_train.
Preparing later_evaluation...
later_evaluation: 1,044,663 records written
later_evaluation: 2,021,954 records written
later_evaluation: 3,000,618 records written
later_evaluation: 4,042,118 records written
later_evaluation: 5,022,031 records written
later_evaluation: 6,063,967 records written
later_evaluation: 7,042,345 records written
later_evaluation: 8,022,212 records written
later_evaluation: 9,000,830 records written
later_evaluation: 10,043,409 r

,cohort,users,sequence_days,channels,users_with_logs,users_without_logs,recorded_user_days,invalid_duration_days
0,historical_train,882127,30,9,667477,214650,11326275,2384
1,later_evaluation,970960,30,9,758941,212019,12682786,3052


In [8]:
REPORT_DIR.mkdir(parents=True, exist_ok=True)

sequence_summary.to_csv(
    REPORT_DIR / "sequence_preparation_summary.csv",
    index=False,
)

sequence_config = {
    "sequence_directory": "data/processed/gru_sequences_v1",
    "sequence_length": SEQUENCE_LENGTH,
    "sequence_channels": sequence_channels,
    "sequence_dtype": "float32",
    "numeric_transform": "log1p",
    "transaction_columns": transaction_columns,
    "transaction_missing_values": "retained for training-only preprocessing",
    "user_order": "msno ascending within each cohort",
    "missing_day_encoding": "all zeros with log_present=0",
    "invalid_duration_encoding": (
        "duration value=0, log_present=1, duration_valid=0; "
        "song counts retained"
    ),
    "all_labeled_users_retained": True,
    "toy_alignment_checks_passed": True,
    "source_record_counts_verified": True,
    "learned_preprocessing_fitted": False,
    "cohorts": cohort_specs,
    "pytorch_version": torch.__version__,
}

(REPORT_DIR / "sequence_preparation_config.json").write_text(
    json.dumps(sequence_config, indent=2) + "\n",
    encoding="utf-8",
)

print("Sequence preparation summary and configuration saved.")

Sequence preparation summary and configuration saved.


## Sequence Preparation for GRU

Constructed chronological 30-day listening sequences for all 882,127 historical training users and 970,960 later evaluation users.

Each sequence contains nine channels: six log-transformed song counts, log-transformed valid listening duration, log presence, and duration validity. Seven transaction features are saved separately as additional model inputs.

Missing days have zero values and a log-presence indicator of zero. Recorded days with invalid duration retain their song counts and have duration validity set to zero.

Sequences, transaction inputs, labels, and user metadata share the same user ordering. Alignment checks and reconciliation against extracted daily-record counts passed.

Arrays are stored locally as memory-mapped NumPy files and excluded from Git. No learned preprocessing parameters have been fitted at this stage.

See [the preparation notebook](notebooks/15_sequence_preparation.ipynb), [summary](reports/sequence_preparation_summary.csv), and [input configuration](reports/sequence_preparation_config.json).